# 光明翻译器 — 多模型 LoRA/QLoRA 微调 Notebook

本 Notebook 引导你逐步完成 LoRA 微调，使模型学会将 Python 代码翻译为光明 v3.2 代码。

## 支持模型
| 模型 | 参数量 | LoRA 显存 | QLoRA 显存 | 适用场景 |
|------|--------|-----------|------------|----------|
| **Qwen3.5-2B** | 2B | ~5 GB | ~3 GB | 开发调试首选，飞快 |
| **Qwen3-8B** | 8B | ~22 GB | ~8 GB | 生产部署，效果最强 |

**推荐工作流**：先用 Qwen3.5-2B 快速验证数据/prompt 质量（~10分钟），确认效果后切 Qwen3-8B 做生产级微调。

## 显存需求
| 模型 + 模式 | 显存 | 适用显卡 |
|-------------|------|----------|
| Qwen3.5-2B LoRA BF16 | ~5 GB | GTX 1660 / RTX 3060 |
| Qwen3.5-2B QLoRA 4bit | ~3 GB | 几乎任何 GPU |
| Qwen3-8B LoRA BF16 | ~22 GB | RTX 3090/4090/A100 |
| Qwen3-8B QLoRA 4bit | ~8 GB | RTX 4060/4070 |

## Cell 1: 环境安装

首次运行时安装依赖。如果你已有环境，可跳过此 Cell。

In [ ]:
# 安装 LLaMA-Factory（二选一）

# 方式1: 从 PyPI 安装
!pip install llamafactory transformers accelerate peft

# 方式2: 从源码安装（推荐，获取最新功能）
# !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
# !cd LLaMA-Factory && pip install -e ".[torch,metrics]"

# QLoRA 4bit 量化需要（如使用 QLoRA 请取消注释）
# !pip install bitsandbytes

print("安装完成！")

## Cell 2: 环境检查

确认 GPU、PyTorch 和 LLaMA-Factory 是否就绪。

In [ ]:
import sys
print(f"Python: {sys.version}")

# 检查 PyTorch + CUDA
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
        print(f"显存: {mem:.1f} GB")
        if mem >= 22:
            print("✓ 显存充足，推荐 LoRA BF16 模式")
        elif mem >= 8:
            print("⚠ 推荐使用 QLoRA 4bit 模式")
        else:
            print("⚠ 显存不足 8GB，需要 CPU offload")
except ImportError:
    print("✗ PyTorch 未安装")

# 检查 LLaMA-Factory
try:
    import llamafactory
    print(f"✓ LLaMA-Factory 已安装")
except ImportError:
    print("✗ LLaMA-Factory 未安装，请运行 Cell 1")

# 检查 bitsandbytes
try:
    import bitsandbytes
    print(f"✓ bitsandbytes {bitsandbytes.__version__}（QLoRA 可用）")
except ImportError:
    print("ℹ bitsandbytes 未安装（QLoRA 不可用，LoRA BF16 不受影响）")

## Cell 3: 配置参数

在这里修改训练参数。初学者只需关注 `USE_QLORA` 和 `MODEL_PATH`。

In [ ]:
import os

# ══════════════════════════════════════════════════════════
# 核心配置（新手只需改这几个）
# ══════════════════════════════════════════════════════════

# 模型预设：
#   'qwen3.5-2b' — 2B 轻量模型，开发调试首选（LoRA ~5GB / QLoRA ~3GB）
#   'qwen3-8b'   — 8B 大模型，生产部署首选（LoRA ~22GB / QLoRA ~8GB）
MODEL_PRESET = 'qwen3.5-2b'

# 预设配置（自动根据 MODEL_PRESET 填充，也可手动覆盖）
PRESETS = {
    'qwen3.5-2b': {
        'model_path': 'Qwen/Qwen3.5-2B-Instruct',
        'output_dir': os.path.expanduser('~/light_lora_output_2b'),
        'batch_size': 4,
        'grad_accum': 4,
        'lr': 2e-4,
        'lora_rank': 16,
        'template': 'qwen3',
    },
    'qwen3-8b': {
        'model_path': 'Qwen/Qwen3-8B-Instruct',
        'output_dir': os.path.expanduser('~/light_lora_output_8b'),
        'batch_size': 2,
        'grad_accum': 8,
        'lr': 1e-4,
        'lora_rank': 16,
        'template': 'qwen3',
    },
}

preset = PRESETS[MODEL_PRESET]

# 是否使用 QLoRA 4bit 量化
#   2B 模型通常不需要（~5GB 就够），8B 模型显存不够时开启
USE_QLORA = False

# 以下参数会被预设填充，也可以手动覆盖
MODEL_PATH = preset['model_path']
OUTPUT_DIR = preset['output_dir']
BATCH_SIZE = preset['batch_size']
GRAD_ACCUM = preset['grad_accum']
LEARNING_RATE = preset['lr']
LORA_RANK = preset['lora_rank']
TEMPLATE = preset['template']

# ══════════════════════════════════════════════════════════
# 训练参数（一般不需要改）
# ══════════════════════════════════════════════════════════

EPOCHS = 3                  # 训练轮数（3 轮通常足够）
LORA_ALPHA = LORA_RANK * 2  # LoRA 缩放系数
MAX_SEQ_LEN = 1024          # 最大序列长度
WARMUP_RATIO = 0.05        # 预热比例

# ══════════════════════════════════════════════════════════
# 数据路径
# ══════════════════════════════════════════════════════════

DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.',
                             'tools', 'ai_copilot', 'sft_dataset.jsonl')
if not os.path.isfile(DATASET_PATH):
    DATASET_PATH = os.path.join('tools', 'ai_copilot', 'sft_dataset.jsonl')

print(f"配置摘要：")
print(f"  预设: {MODEL_PRESET}")
print(f"  模式: {'QLoRA 4bit' if USE_QLORA else 'LoRA BF16'}")
print(f"  模型: {MODEL_PATH}")
print(f"  输出: {OUTPUT_DIR}")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  学习率: {LEARNING_RATE}")
print(f"  等效批大小: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  数据: {DATASET_PATH}")

## Cell 4: 检查训练数据

查看数据集的基本统计信息。

In [ ]:
import json
from collections import Counter

categories = []
total = 0

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        categories.append(item.get('category', 'unknown'))
        total += 1

cat_counts = Counter(categories)

print(f"数据集统计：")
print(f"  总样本数: {total}")
print(f"  类别数: {len(cat_counts)}")
print(f"\n类别分布：")
for cat, count in cat_counts.most_common():
    bar = '█' * (count // 3)
    print(f"  {cat:6s}: {count:3d} {bar}")

# 预览几条样本
print(f"\n样本预览：")
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        item = json.loads(line.strip())
        print(f"\n  [{i+1}] {item.get('category', '?')}")
        print(f"      输入: {item.get('input', '')[:80]}")
        print(f"      输出: {item.get('output', '')[:80]}")

## Cell 5: 准备数据（转 ShareGPT 格式）

LLaMA-Factory 推荐使用 ShareGPT 格式（多轮对话）。这里将 Alpaca 格式转为 ShareGPT，并添加 system prompt。

In [ ]:
import json
import os

SYSTEM_PROMPT = (
    "你是光明（LightLang）编程语言 v3.2 的翻译专家。"
    "光明是一种中文编程语言，使用中文关键字。"
    "你的任务是将 Python 代码翻译为光明 v3.2 代码。"
    "注意暗坑：长度()不可用用len()、列表索引赋值用方括号语法、"
    "变量名不能与内建函数同名、类系统需LLVM后端。"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
sharegpt_path = os.path.join(OUTPUT_DIR, 'light_sft_sharegpt.jsonl')

count = 0
with open(DATASET_PATH, 'r', encoding='utf-8') as fin, \
     open(sharegpt_path, 'w', encoding='utf-8') as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)

        instruction = item.get('instruction', '将Python代码转为光明代码：')
        code_input = item.get('input', '')
        output = item.get('output', '')

        user_msg = f"{instruction}\n\nPython代码：\n{code_input}" if code_input else instruction

        conv = {
            "conversations": [
                {"from": "system", "value": SYSTEM_PROMPT},
                {"from": "human", "value": user_msg},
                {"from": "gpt", "value": output},
            ]
        }
        fout.write(json.dumps(conv, ensure_ascii=False) + '\n')
        count += 1

print(f"✓ 转换完成: {sharegpt_path} ({count} 条)")

# 预览转换后的样本
with open(sharegpt_path, 'r', encoding='utf-8') as f:
    first = json.loads(f.readline())
    print(f"\n转换后样本预览：")
    for msg in first['conversations']:
        role = msg['from']
        value = msg['value'][:100]
        print(f"  [{role}]: {value}...")

## Cell 6: 注册数据集到 LLaMA-Factory

将数据集注册到 LLaMA-Factory 的 dataset_info.json，使训练命令能找到它。

In [ ]:
import os
import json
import shutil

DATASET_NAME = 'light_v32_sft'

# 查找 LLaMA-Factory 数据目录
lf_data_dir = None
for candidate in [
    os.path.join(os.getcwd(), 'LLaMA-Factory', 'data'),
    os.path.expanduser('~/LLaMA-Factory/data'),
]:
    if os.path.isdir(candidate):
        lf_data_dir = candidate
        break

# 尝试通过 pip 查找
if lf_data_dir is None:
    try:
        import subprocess
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', 'llamafactory'],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            for line in result.stdout.splitlines():
                if line.startswith('Location:'):
                    loc = line.split(':', 1)[1].strip()
                    candidate = os.path.join(loc, 'llamafactory', 'data')
                    if os.path.isdir(candidate):
                        lf_data_dir = candidate
                        break
    except Exception:
        pass

registered = False

if lf_data_dir:
    # 复制数据文件
    target_path = os.path.join(lf_data_dir, 'light_sft_sharegpt.jsonl')
    shutil.copy2(sharegpt_path, target_path)
    print(f"✓ 数据复制到: {target_path}")

    # 更新 dataset_info.json
    info_path = os.path.join(lf_data_dir, 'dataset_info.json')
    info = {}
    if os.path.isfile(info_path):
        with open(info_path, 'r', encoding='utf-8') as f:
            info = json.load(f)

    info[DATASET_NAME] = {
        "file_name": "light_sft_sharegpt.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations"},
    }

    with open(info_path, 'w', encoding='utf-8') as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    print(f"✓ 数据集已注册: {DATASET_NAME}")
    registered = True
else:
    print("⚠ 未找到 LLaMA-Factory 数据目录")
    print("将使用绝对路径方式训练")
    registered = False

print(f"\n数据集引用方式: {'dataset_name' if registered else 'absolute_path'}")

## Cell 7: 生成训练配置并启动训练

这是核心步骤！生成 YAML 配置并开始训练。

预计时间（881 条数据 × 3 epochs）：
- Qwen3.5-2B LoRA BF16: ~10 分钟（任何 GPU）
- Qwen3-8B LoRA BF16: ~30 分钟（RTX 4090）
- Qwen3-8B QLoRA 4bit: ~60 分钟（RTX 4060）

In [ ]:
import yaml

# 生成训练配置
config = {
    # 模型
    'model_name_or_path': MODEL_PATH,
    'trust_remote_code': True,

    # 微调方法
    'stage': 'sft',
    'do_train': True,
    'finetuning_type': 'lora',
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'lora_target': 'q_proj,v_proj,k_proj,o_proj,gate_proj,up_proj,down_proj',
    'lora_dropout': 0.05,

    # 数据集
    'template': TEMPLATE,
    'cutoff_len': MAX_SEQ_LEN,

    # 训练参数
    'output_dir': os.path.join(OUTPUT_DIR, 'checkpoints'),
    'per_device_train_batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'num_train_epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': WARMUP_RATIO,
    'logging_steps': 10,
    'save_steps': 100,
    'save_total_limit': 3,

    # 精度
    'bf16': not USE_QLORA,
    'fp16': False,
    'gradient_checkpointing': True,

    # 其他
    'overwrite_output_dir': False,
    'report_to': 'none',
    'seed': 42,
}

# QLoRA 配置
if USE_QLORA:
    config['quantization_bit'] = 4
    config['quantization_method'] = 'nf4'

# 数据集引用
if registered:
    config['dataset'] = DATASET_NAME
else:
    config['dataset'] = sharegpt_path
    config['dataset_dir'] = os.path.dirname(sharegpt_path)
    config['formatting'] = 'sharegpt'

# 写入配置
yaml_path = os.path.join(OUTPUT_DIR, 'train_config.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)

print(f"✓ 配置已生成: {yaml_path}")
print(f"\n训练配置摘要：")
for key in ['model_name_or_path', 'finetuning_type', 'lora_rank', 'learning_rate',
            'per_device_train_batch_size', 'gradient_accumulation_steps',
            'num_train_epochs', 'bf16']:
    print(f"  {key}: {config.get(key, 'N/A')}")

# ── 开始训练 ──
print(f"\n{'='*60}")
print(f"开始训练...")
print(f"{'='*60}")

import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'llamafactory.cli', 'train', yaml_path],
)

if result.returncode == 0:
    print("\n✓ 训练完成！")
else:
    print(f"\n⚠ 训练返回码: {result.returncode}")
    print(f"请检查上方日志输出。")

## Cell 8: 合并 LoRA 权重

训练完成后，LoRA adapter 是独立的小文件。合并到基础模型后才能独立使用。

In [ ]:
import os
import subprocess

checkpoint_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
merged_dir = os.path.join(OUTPUT_DIR, 'merged')

# 找最新 checkpoint
ckpt_dirs = sorted([
    d for d in os.listdir(checkpoint_dir)
    if d.startswith('checkpoint-') and os.path.isdir(os.path.join(checkpoint_dir, d))
], key=lambda x: int(x.split('-')[-1]) if x.split('-')[-1].isdigit() else 0)

latest_ckpt = os.path.join(checkpoint_dir, ckpt_dirs[-1]) if ckpt_dirs else checkpoint_dir
print(f"最新 checkpoint: {latest_ckpt}")

# 生成合并配置
merge_config = {
    'model_name_or_path': MODEL_PATH,
    'adapter_name_or_path': latest_ckpt,
    'template': 'qwen3',
    'finetuning_type': 'lora',
    'lora_rank': LORA_RANK,
    'export_dir': merged_dir,
    'export_size': 2,
    'export_device': 'cpu',
    'export_legacy_format': False,
}

merge_yaml = os.path.join(OUTPUT_DIR, 'merge_config.yaml')
with open(merge_yaml, 'w', encoding='utf-8') as f:
    yaml.dump(merge_config, f, allow_unicode=True, default_flow_style=False)

print(f"合并配置: {merge_yaml}")

# 执行合并
result = subprocess.run(
    [sys.executable, '-m', 'llamafactory.cli', 'export', merge_yaml],
)

if result.returncode == 0:
    print(f"\n✓ 合并完成: {merged_dir}")
else:
    print(f"\n⚠ 合并返回码: {result.returncode}")
    print(f"手动合并: llamafactory-cli export {merge_yaml}")

## Cell 9: 测试推理效果

用几个 Python 代码片段测试微调后模型的翻译能力。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_dir = os.path.join(OUTPUT_DIR, 'merged')
use_model = merged_dir if os.path.isdir(merged_dir) else MODEL_PATH

print(f"加载模型: {use_model}")
tokenizer = AutoTokenizer.from_pretrained(use_model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    use_model,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()
print("✓ 模型加载完成")

# 测试用例
test_cases = [
    "def add(a, b): return a + b",
    "x = 10\nprint(x)",
    "for i in range(5): print(i)",
    "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n-1)",
    "x += 1",
    "while True:\n    if x == 0:\n        break\n    x -= 1",
]

SYSTEM_PROMPT = (
    "你是光明编程语言v3.2的翻译专家。"
    "将Python代码翻译为光明v3.2代码。"
)

print(f"\n{'='*60}")
print(f"推理测试")
print(f"{'='*60}")

for i, python_code in enumerate(test_cases, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"将Python代码转为光明代码：\n\nPython代码：\n{python_code}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n测试 {i}:")
    print(f"  Python: {python_code}")
    print(f"  光明:   {response}")

## Cell 10: 部署指南

训练完成后，有三种部署方式：

In [ ]:
merged_dir = os.path.join(OUTPUT_DIR, 'merged')

print("="*60)
print("部署方式")
print("="*60)

print(f"""
1. vLLM 部署（推荐，生产级）
   pip install vllm
   vllm serve {merged_dir} --port 8000

2. Ollama 部署（本地推理）
   # 先转换为 GGUF 格式
   pip install llama-cpp-python
   # 然后导入 Ollama
   ollama create light-translator -f Modelfile

3. API 服务（开发调试）
   python -m llamafactory.cli api {merged_dir}

4. 集成到光明管线
   # 将微调后的模型路径配置到 light ai generate
   light ai generate "排序算法" --model-path {merged_dir}
""")

## 附录: Loss 曲线诊断

训练时如果 loss 不降，参考以下排查指南：

| 症状 | 可能原因 | 解决方案 |
|------|----------|----------|
| loss 不下降 | lr 太小 | 提高到 1e-4 或 2e-4 |
| loss 震荡剧烈 | lr 太大 | 降到 5e-5 |
| loss 下降后反弹 | 过拟合 | 减少 epochs 或加大 lora_dropout |
| OOM 爆显存 | batch 太大 | 减小 batch_size 或用 QLoRA |
| 训练很慢 | 未开 gradient_checkpointing | 确认配置中 gradient_checkpointing: true |

关键经验：
- LoRA/QLoRA 的学习率要比全参微调高 5-10 倍（1e-4 而非 2e-5）
- lora_rank 不是越大越好，16 通常足够
- 梯度累积步数要够大，等效 batch_size 建议至少 8